In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

df = pd.read_csv("data/application_train_features.csv")

features = [
    # Engineered features
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_INCOME",
    "CREDIT_TO_GOODS",
    "EMPLOYMENT_RATIO",
    "INCOME_PER_PERSON",
    "CHILDREN_RATIO",

    # External credit bureau scores — strongest predictors
    "EXT_SOURCE_1",
    "EXT_SOURCE_1_MISSING",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",

    # Original strong numerics
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_EMPLOYED_MISSING",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",

    # Encoded categoricals
    "NAME_CONTRACT_TYPE_ENC",
    "CODE_GENDER_ENC",
    "NAME_INCOME_TYPE_ENC",
    "NAME_EDUCATION_TYPE_ENC",
    "NAME_FAMILY_STATUS_ENC",
    "OCCUPATION_TYPE_ENC",

    # Aggregated flags
    "DOCUMENT_COUNT",
    "CONTACT_REACHABILITY",
]

X = df[features]
y = df["TARGET"]

print("Features shape:", X.shape)
print("Default rate:", y.mean().round(4))

Features shape: (307511, 26)
Default rate: 0.0807


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,        # 80% train, 20% validate
    random_state=42,      # reproducible split
    stratify=y            # keeps 8% default rate in both splits
)

print("Training set:  ", X_train.shape)
print("Validation set:", X_val.shape)
print("\nDefault rate in train:", y_train.mean().round(4))
print("Default rate in val:  ", y_val.mean().round(4))

Training set:   (246008, 26)
Validation set: (61503, 26)

Default rate in train: 0.0807
Default rate in val:   0.0807


In [8]:
# scale_pos_weight fixes the 92/8 class imbalance we found in Cell 2
# It tells XGBoost: "pay 11x more attention to defaulters"
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale:.2f}  ← this fixes class imbalance")

model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale,   # class imbalance fix
    eval_metric="auc",
    early_stopping_rounds=50, # stops if no improvement for 50 rounds
    random_state=42,
    n_jobs=-1                 # use all CPU cores
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100               # print AUC every 100 rounds
)

scale_pos_weight: 11.39  ← this fixes class imbalance
[0]	validation_0-auc:0.71704
[100]	validation_0-auc:0.75121
[200]	validation_0-auc:0.75385
[300]	validation_0-auc:0.75430
[324]	validation_0-auc:0.75413


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

In [9]:
# Predict probabilities (not just 0/1)
y_pred_proba = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred_proba)
print(f"✅ Validation AUC-ROC: {auc:.4f}")
print("   (Target: > 0.75  |  Good model: > 0.78  |  Great model: > 0.80)")

# Show a sample of predictions
sample = X_val.head(5).copy()
sample["ACTUAL_DEFAULT"] = y_val.head(5).values
sample["RISK_SCORE"] = y_pred_proba[:5].round(3)
print("\nSample predictions:")
print(sample[["ACTUAL_DEFAULT", "RISK_SCORE"]])

✅ Validation AUC-ROC: 0.7544
   (Target: > 0.75  |  Good model: > 0.78  |  Great model: > 0.80)

Sample predictions:
        ACTUAL_DEFAULT  RISK_SCORE
256571               0       0.479
191493               0       0.304
103497               0       0.786
130646               0       0.247
211898               0       0.500


In [10]:
import pickle

with open("data/credit_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save feature list too — needed later for explainability
with open("data/feature_list.pkl", "wb") as f:
    pickle.dump(features, f)

print("✅ Model saved to data/credit_model.pkl")
print("✅ Features saved to data/feature_list.pkl")

✅ Model saved to data/credit_model.pkl
✅ Features saved to data/feature_list.pkl
